In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

MODEL_NAME = "facebook/nllb-200-distilled-600M"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

def translate(texts, src_lang, tgt_lang, batch_size=32):
    tokenizer.src_lang = src_lang
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    outputs = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_new_tokens=128
            )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        outputs.extend(decoded)

    return outputs

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving youtube_test_en.csv to youtube_test_en.csv
Saving youtube_train_en.csv to youtube_train_en.csv
Saving youtube_val_en.csv to youtube_val_en.csv
User uploaded file "youtube_test_en.csv" with length 91961 bytes
User uploaded file "youtube_train_en.csv" with length 399099 bytes
User uploaded file "youtube_val_en.csv" with length 87129 bytes


In [ ]:
# Load your YouTube translated file
df = pd.read_csv("youtube_train_en.csv")

# Take small sample (important: don't run on full dataset)
sample_df = df.sample(50, random_state=42)

# Step 1: English → Nepali (back translation)
back_translated = translate(
    sample_df["text_en"].tolist(),
    src_lang="eng_Latn",
    tgt_lang="npi_Deva"
)

sample_df["back_ne"] = back_translated

100%|██████████| 2/2 [00:01<00:00,  1.25it/s]


In [ ]:
sample_df[["text", "text_en", "back_ne"]].head(10)

,text,text_en,back_ne
390,sojha नेपाली हरू लाई आत्याचार गर्ने तिमी हरू क...,Kulat to you who have persecuted sojha Nepalis.,"कुलत तिमीहरूलाई, जसले हालसम्मका नेपालीहरूलाई स..."
247,रास्ट पति को यहि हो बोल्ने सबे थुक यक नारि भयर...,If you tell a woman to marry a ruthless husban...,यदि तपाईंले एक महिलालाई क्रूर पतिसँग विवाह गर्...
260,प्रचन्ड ले नै बिगारेको छ अहिले नेपाल भर्स्ट्चा...,"Pradhan has already been corrupted, but even t...","प्रचण्ड पहिले नै भ्रष्ट भएका छन्, तर नेपालले ब..."
155,यो कुकुर को छाऊरो ले गलत भन्दैछ यो कुकुर भने क...,This dog is a dog that is unemployed and is no...,यो कुकुर बेरोजगार कुकुर हो र नेपालीहरूको लागि ...
984,यो पल हाम्रो लागि महत्त्व पुर्ण छ ।,This moment is very important to us.,यो क्षण हाम्रो लागि धेरै महत्त्वपूर्ण छ।
413,यस्ता अविवेकी पत्रकार सङ्ग के अन्तर्वार्ता दिनु ।,Interview such a foolish press association.,यस्तो मूर्ख प्रेस एसोसिएशनसँग अन्तर्वार्ता लिन...
802,पत्रकार भएसी जस्तो अबस्था मा सत्यतथ्य कुरा देख...,"Just like a journalist, you should be able to ...","पत्रकार जस्तै, तपाईं पनि समयमै सत्य बताउन सक्ष..."
58,सुशील सर आईन्धा यो पुलिस डिपार्टमेन्ट को कुनै ...,Sushil Sir Aisha should not have invited this ...,सुशील सर ऐशाले यस प्रहरी विभागलाई कुनै पनि पोश...
752,यो केशब बडालभन्ने जन्तु लाई कुनै पनि tv च्यानल...,Never speak of this Keshb Badalabhane Jantu on...,कुनै पनि टेलिभिजन च्यानलमा यस केशब बजालाभाने ज...
901,यस्ता प्रश्न म लाई नसोध्नु पो भन्छन् कुकुर जस्...,"Don't ask me such questions, but do they say t...","मलाई त्यस्ता प्रश्नहरू न सोध्नुहोस्, तर के तिन..."


In [ ]:
sample_df.to_csv("youtube_train_en_back_translated.csv", index=False)
print("Back-translated data saved to youtube_train_en_back_translated.csv")

Back-translated data saved to youtube_train_en_back_translated.csv
